Let's use Apache Spark framwork instead of Julia for data processing (note: Spark is not optimized for GPU trainings.)


In [0]:
# This function calculates the RCA (Revealed Comparative Advantage)
# https://unctadstat.unctad.org/EN/RcaRadar.html
def RCA(data):
    # 1. Aggregations on exporters
    exporter_product = data.groupBy("year", "hsCode", "exporter").agg(
        F.sum("value").alias("TotalExportofCmdbyPartner")
    )
    
    exporter = data.groupBy("year", "exporter").agg(
        F.sum("value").alias("TotalExportbyPartner")
    )
    
    # 2. Aggregations at Wolrd export level
    alberta_excluded = data.filter(F.col("exporter") != 9999)
    
    world_product = alberta_excluded.groupBy("year", "hsCode").agg(
        F.sum("value").alias("WorldExportofCmd")
    )
    
    world = alberta_excluded.groupBy("year").agg(
        F.sum("value").alias("WorldTotalExport")
    )
    
    # 3. merge the data
    data = data.join(exporter_product, on=["year", "hsCode", "exporter"], how="left")
    data = data.join(exporter, on=["year", "exporter"], how="left")
    data = data.join(world_product, on=["year", "hsCode"], how="left")
    data = data.join(world, on=["year"], how="left")
    
    # 4. RCA formula: (Partner_Cmd_Export / Partner_Total_Export) / (World_Cmd_Export / World_Total_Export)
    data = data.withColumn(
        "Partner_Revealed_Comparative_Advantage",
        (F.col("TotalExportofCmdbyPartner") / F.col("TotalExportbyPartner")) /
        (F.col("WorldExportofCmd") / F.col("WorldTotalExport"))
    )
    
    return data

# This function calculates the Theil Concentration Index for the exporter
# https://www.un.org/development/desa/dpad/wp-content/uploads/sites/45/CDP-bp-2023-59.pdf
def calculate_theil_exporter_concentration(df):
    # It would have been better to define n, m for each year
    # However, it does not matter as n, m dont change from a year to another except for 1 unit! (from 2013 to 2023 investigated)
    # 1. Calculate n (unique hsCodes) and m (unique importers)
    # Note: In Spark, we collect these as scalars to use in formulas
    n = df.select("hsCode").distinct().count()
    m = df.select("importer").distinct().count()
    
    print(f"Minimum and Maximum Theil Exporter/Importer Concentration Index: [0, {round(math.log(n*m), 2)}]")

    # 2. total_export_by_partner (Year, Exporter)
    total_export_by_partner = df.groupBy("year", "exporter").agg(
        F.sum("value").alias("TotalExportbyPartner"))

    # 3. total_export_by_partner_of_cmd (Year, Exporter, hsCode)
    total_export_by_partner_of_cmd = df.groupBy("year", "exporter", "hsCode").agg(
        F.sum("value").alias("TotalExportbyPartnerofCmd")).join(
            total_export_by_partner, on=["year", "exporter"], how="left")

    # 4. Calculate x_k and T_p (Product Concentration)
    # T_p = Sum(x_k * log(x_k)) + log(n)
    total_export_by_partner_of_cmd = total_export_by_partner_of_cmd.withColumn(
        "x_k", F.col("TotalExportbyPartnerofCmd") / F.col("TotalExportbyPartner")
    ).withColumn(
        "x_k_times_ln_x", F.col("x_k") * F.log(F.col("x_k"))
    )

    t_p = total_export_by_partner_of_cmd.groupBy("year", "exporter") \
        .agg(F.sum("x_k_times_ln_x").alias("sum_xk_ln_xk")) \
        .withColumn("T_p", F.col("sum_xk_ln_xk") + math.log(n)) \
        .select("year", "exporter", "T_p")

    # 5. Calculate T_m (Market Concentration)
    # T_m = Sum over k of [ x_k * Sum over j of (x_jk * log(x_jk * m)) ]
    
    # x_jk = value / TotalExportbyPartnerofCmd
    t_m_prep = df.join(
        total_export_by_partner_of_cmd.select("year", "exporter", "hsCode", "TotalExportbyPartnerofCmd"),
        on=["year", "exporter", "hsCode"],
        how="left"
    )

    t_m = t_m_prep.withColumn("x_j_k", F.col("value") / F.col("TotalExportbyPartnerofCmd")) \
        .withColumn("x_j_k_times_ln_xm", F.col("x_j_k") * F.log(F.col("x_j_k") * m)) \
        .groupBy("year", "exporter", "hsCode") \
        .agg(F.sum("x_j_k_times_ln_xm").alias("A"))

    t_m = t_m.join(total_export_by_partner_of_cmd.select("year", "exporter", "hsCode", "x_k"), on=["year", "exporter", "hsCode"], how="left").withColumn("B", F.col("A") * F.col("x_k")).groupBy(
        "year", "exporter") \
                  .agg(
                      F.sum("B").alias("T_m")
    )

    # 6. Final Join and Calculation
    result_df = df.join(t_p, on=["year", "exporter"], how="left") \
                  .join(t_m, on=["year", "exporter"], how="left") \
                  .withColumn("Theil_Exporter_Concentration", F.col("T_p") + F.col("T_m"))

    return result_df

# This function calculates the Theil Concentration Index for the importer
# https://www.un.org/development/desa/dpad/wp-content/uploads/sites/45/CDP-bp-2023-59.pdf
def calculate_theil_importer_concentration(df):
    # 1. Exclude Alberta (exporter != 9999)
    alberta_excluded = df.filter(F.col("exporter") != 9999)

    # 2. Compute n and m
    n = alberta_excluded.select("hsCode").distinct().count()
    m = alberta_excluded.select("exporter").distinct().count()

    nm = n * m

    # 3. Compute psi = sum(value) per (year, importer)
    psi = (
        alberta_excluded
        .groupBy("year", "importer")
        .agg(F.sum("value").alias("psi"))
        .withColumn("psi", F.col("psi") / nm)
    )

    # 4. Join psi back
    psi_joined = alberta_excluded.join(psi, on=["year", "importer"], how="left")

    # 5. Compute x = (value/psi) * log(value/psi)
    ratio = F.col("value") / F.col("psi")
    psi_joined = psi_joined.withColumn("x", ratio * F.log(ratio))

    # 6. Aggregate x per importer-year
    x_agg = (
        psi_joined
        .groupBy("year", "importer")
        .agg(F.sum("x").alias("x"))
        .withColumn("Theil_Importer_Concentration", F.col("x") / nm)
        .select("year", "importer", "Theil_Importer_Concentration")
    )

    # 7. Join back to original data
    result = df.join(x_agg, on=["year", "importer"], how="left")

    return result

# This function calculates the Product Complexity Index
# https://atlas.hks.harvard.edu/glossary
# https://www.pnas.org/doi/epdf/10.1073/pnas.0900943106
def Product_Complexity_Index(df):
    pritn("Product Complexity Index")
    return

# This function calculates the Product Relatedness Index
# Prodcut relatedness is defined for a pair of products! Relatedness(Produckt_i, Product_j)!
# https://www.cepii.fr/PDF_PUB/wp/2012/wp2012-27.pdf
def Product_Relatedness_Index(df):
    print("Product Relatedness Index")
    return

def calculate_trade_complementarity(df):
    # 1. Exclude Alberta for importer-side totals
    importer_data = df.filter(F.col("exporter") != 9999)

    # 2. Total imports by importer (year, importer)
    importer_totals = (
        importer_data
        .groupBy("year", "importer")
        .agg(F.sum("value").alias("TotalImportbyImporter"))
    )

    # 3. Total exports by exporter (year, exporter)
    exporter_totals = (
        df.groupBy("year", "exporter")
          .agg(F.sum("value").alias("TotalExportbyExporter"))
    )

    # 4. Join totals back to main dataset
    df2 = (
        df.join(importer_totals, on=["year", "importer"], how="left")
          .join(exporter_totals, on=["year", "exporter"], how="left")
    )

    # 5. Compute Trade_Complementarity = abs((a/b) - (c/d))
    df2 = df2.withColumn(
        "Trade_Complementarity",
        F.abs(
            (F.col("TotalImportofCmdbyReporter") / F.col("TotalImportbyImporter")) -
            (F.col("TotalExportofCmdbyPartner") / F.col("TotalExportbyExporter"))
        )
    )

    # 6. Drop intermediate totals
    df2 = df2.drop("TotalImportbyImporter", "TotalExportbyExporter")

    return df2



#### HS4 Basket Distance #######
# Calculates the HS4 Basket Distance
# Also trasforms the HS6 level data to HS4 level
def calculate_HS4_basket_distance(df):
    # Aggregate total exports at the Year–Exporter–HS4 level
    # Aggregate total exports at the Year–Importer–HS4 level
    # Create a Year–Exporter–Importer–HS6 table
    # Join the two aggregated tables with the original data to obtain:
    #   Year, Exporter, Importer, HS4, HS6, value,
    #   total_export_HS4_by_exporter,
    #   total_export_HS4_by_importer
    # Compute the HS6 share within each HS4 for both exporter and importer
    # Calculate the Euclidean distance for each HS4–Exporter–Importer combination
    # Merge the Euclidean distance results back into the original table
    # Call it HS4 Basket Distance



    df = df.withColumn("HS6", F.lpad(F.col("hsCode"), 6, "0"))
    df = df.withColumn("HS4", F.substring(F.col("HS6"), 1, 4))
    data = df
    exporter_product_year = data.groupBy("year", "HS4", "exporter").agg(
        F.sum("value").alias("total_export_HS4_by_exporter")
    )
    importer_product_year = data.groupBy("year", "HS4", "importer").agg(
        F.sum("value").alias("total_import_HS4_by_importer")
    )
    all_exporters = data.select("exporter").distinct()
    all_importers = data.select("importer").distinct()
    all_years = data.select("year").distinct()
    all_hs6 = data.select("HS6").distinct()
    exporter_importer_product_year = (
        all_years.crossJoin(all_hs6)
                .crossJoin(all_exporters).crossJoin(all_importers)
    )
    data = data.join(exporter_product_year.select(["year", "HS4", "exporter", "total_export_HS4_by_exporter"]), ["year", "HS4", "exporter"], how = "left")
    data = data.join(importer_product_year.select(["year", "HS4", "importer", "total_import_HS4_by_importer"]), ["year", "HS4", "importer"], how = "left")
    data = data.withColumn("HS4_share_by_exporter", F.col("value") / F.col("total_export_HS4_by_exporter"))
    data = data.withColumn("HS4_share_by_importer", F.col("value") / F.col("total_import_HS4_by_importer"))
    data = data.groupBy("year", "HS4", "exporter", "importer").agg(
        F.sqrt(
            F.sum((F.col("HS4_share_by_exporter") - F.col("HS4_share_by_importer"))**2)
                    ).alias("HS4_Basket_Distance")
    )

    # HS6 codes transformed to HS4 codes
    df = df.withColumn("hsCode", F.substring(F.lpad(F.col("hsCode"), 6, "0"), 1, 4))

    # # Moving to HS4 codes level:
    df = df.groupBy("year", "hsCode", "exporter", "importer").agg(
            F.sum("value").alias("value"),
            F.sum("quantity").alias("quantity"),
            # Count how many nulls exist in the quantity column for this group
            F.count(F.when(F.col("quantity").isNull(), 1)).alias("null_count")
        ).withColumn("AvgUnitPrice", 
            F.when(F.col("null_count") > 0, None) # If any nulls found, return None
            .otherwise(F.col("value") / F.col("quantity"))
        ).drop("null_count", "quantity")

    data = data.withColumnRenamed("HS4", "hsCode")
    df = df.join(data, on=["year", "hsCode", "exporter", "importer"], how = "left")
    df = df.drop("HS6")

    return df



In [0]:
import os
import json
import regex as re
from pyspark.sql import functions as F
import math

# Setting up the directories, sparks, root_path
cwd = os.getcwd()
with open(f"{cwd}/model_parameters.json", "r") as f:
    model_parameters = json.load(f)

# key vault
storage_account = "jetitih0018dlad91"
scope_name = "storage-account-kv"
secret_name = "storage-to-databricks-connect-kv"
secret_keys = dbutils.secrets.get(scope=scope_name, key=secret_name)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    secret_keys
)

root_path = f"{model_parameters['root_path']}"
root_path = root_path + "/CEPII_old"


# Reading the CEPII files according to the parameters
folders = [f.path for f in dbutils.fs.ls(root_path)]
folders = [f for f in folders if "BACI_HS12" in f]
folders = [f for f in folders if "alberta" in f]
folders = [f for f in folders if int(re.search(r"Y(\d{4})", f).group(1)) >= model_parameters['Start_year']]
folders = [f for f in folders if int(re.search(r"Y(\d{4})", f).group(1)) <= model_parameters['End_year']]

# Reading the CEPII files
df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(folders)
df = df.withColumnsRenamed({"t": "year", "i": "exporter", "j": "importer", "k": "hsCode", "v": "value", "q": "quantity"})
print('before dropping the duplicates:', (df.count(), len(df.columns)))
df = df.dropDuplicates()
print('after dropping the duplicates:', (df.count(), len(df.columns)))
df = df[df["value"] != 0]
print('after dropping the value=0s:', (df.count(), len(df.columns)))

# Change the quantity to missing values if equal to 0
df = df.withColumn("quantity", F.when(F.col("quantity") == 0, None).otherwise(F.col("quantity")))


############################### 
#### Calculate the HS4 Basket Distance #######
#### Move to HS4 Level ####
df = calculate_HS4_basket_distance(df)
print('after moving to HS4 level:', (df.count(), len(df.columns)))
###############################


Imports = df
# Excluding Alberta from the ImportsfromWorld dataset
ImportsfromWorld = Imports.filter(F.col("exporter") != 9999)
ImportsfromWorld = ImportsfromWorld.withColumn("UnitValueTimesvalue", F.col("value") * F.col("AvgUnitPrice") )
# ImportsfromWorld = ImportsfromWorld.groupBy("year", "hsCode", "importer").agg(
#                         F.sum("value").alias("value"), 
#                         F.sum("UnitValueTimesvalue").alias("UnitValueTimesvalue")
#                         )

ImportsfromWorld = ImportsfromWorld.groupBy("year", "hsCode", "importer").agg(
        F.sum("value").alias("value"),
        F.sum("UnitValueTimesvalue").alias("UnitValueTimesvalue"),
        # Count how many nulls exist in the quantity column for this group
        F.count(F.when(F.col("UnitValueTimesvalue").isNull(), 1)).alias("null_count")
    ).withColumn("AvgUnitPriceofImporterFromWorld", 
        F.when(F.col("null_count") > 0, None) # If any nulls found, return None
        .otherwise(F.col("UnitValueTimesvalue") / F.col("value"))
    ).drop("null_count", "UnitValueTimesvalue")


# Taking care of the total export dataset
ExportstoWorld = df
ExportstoWorld = ExportstoWorld.withColumn("UnitValueTimesvalue", F.col("value") * F.col("AvgUnitPrice") )
# ExportstoWorld = ExportstoWorld.groupBy("year", "hsCode", "exporter").agg(
#                         F.sum("value").alias("value"), 
#                         F.sum("UnitValueTimesvalue").alias("UnitValueTimesvalue")
#                         )

ExportstoWorld = ExportstoWorld.groupBy("year", "hsCode", "exporter").agg(
        F.sum("value").alias("value"),
        F.sum("UnitValueTimesvalue").alias("UnitValueTimesvalue"),
        # Count how many nulls exist in the quantity column for this group
        F.count(F.when(F.col("UnitValueTimesvalue").isNull(), 1)).alias("null_count")
    ).withColumn("AvgUnitPriceofExporterToWorld", 
        F.when(F.col("null_count") > 0, None) # If any nulls found, return None
        .otherwise(F.col("UnitValueTimesvalue") / F.col("value"))
    ).drop("null_count", "UnitValueTimesvalue")

ExportstoWorld = ExportstoWorld.withColumnRenamed("value", "TotalExportofCmdbyExporter")
ImportsfromWorld = ImportsfromWorld.withColumnRenamed("value", "TotalImportofCmdbyReporter")


# Merging df, ExportstoWorld and ImportsfromWorld AS Trade_data
Trade_data = df.join(ImportsfromWorld, ["year", "hsCode", "importer"], how = "left")
Trade_data = Trade_data.join(ExportstoWorld, ["year", "hsCode", "exporter"], how = "left")
Trade_data = Trade_data.orderBy("year", "importer", "exporter", "hsCode")





audit = Trade_data.select(
    # 1. Total count of rows with nulls in TotalImportofCmdbyReporter
    F.count(F.when(F.col("TotalImportofCmdbyReporter").isNull(), 1)).alias("null_count"),
    
    # 2. Total count of rows where exporter is 9999
    F.count(F.when(F.col("exporter") == 9999, 1)).alias("exporter_9999_count"),
    
    # 3. Count of UNIQUE exporters that have at least one null record
    F.count_distinct(
        F.when(F.col("TotalImportofCmdbyReporter").isNull(), F.col("exporter"))
    ).alias("unique_exporters_with_nulls")
                                        ).collect()[0]

a1= audit.null_count
a2 = audit.exporter_9999_count
a3 = audit.unique_exporters_with_nulls
print('Number of economies with TotalImportofCmdbyReporter missing (if 1 then it is just Alberta): ', a3)
print('Ratio of Alberta exports with NULL for TotalImportofCmdbyReporter to total rows of Alberta exports:', a1/a2)

print('Alberta is the only country with missing:', Trade_data.filter(F.col("TotalImportofCmdbyReporter").isNull()).select("exporter").distinct().collect())


# As you can see, for Alberta exports, there are some missing values for TotalImportofCmdbyReporter
# This means while statCan reports Alberta's exports to the world, CEPII does not report imports of these commodities from the world for that country, year.


#print(Trade_data.filter(F.col("TotalImportofCmdbyReporter").isNull()).count()/Trade_data.filter(F.col("exporter") == 9999).count())


# As you can see this is less than 5% of the data and  we have not yet excluded the unwanted countries. 
# Moreoever, this is not going to be part of our training data, so we can ignore it for now.
# For now, I just use the value of Alberta's exports as the value of country's imports.
Trade_data = Trade_data.withColumn("TotalImportofCmdbyReporter", F.when(F.col("TotalImportofCmdbyReporter").isNull(), F.col("value")).otherwise(F.col("TotalImportofCmdbyReporter")))
# NOtice that Alberta's exports are all excluded whenever I am calculating a feature for the importers to prevent the double counting of Alberta and Canada's exports.
# So this would not affect the RCA or Theil Concentration Index calculations of the importers.
# IF not fixed here, we would run into nan RCA and other feature values.

In [0]:
audit

In [0]:
Trade_data2 = RCA(Trade_data)
Trade_data3 = calculate_theil_exporter_concentration(Trade_data2)
Trade_data4 = calculate_theil_importer_concentration(Trade_data3)
Trade_data5 = calculate_trade_complementarity(Trade_data4)


Trade_data5 = Trade_data5.select(
    "year",
    "importer",
    "exporter",
    "hsCode",
    "value",
    "AvgUnitPrice",
    "AvgUnitPriceofImporterFromWorld",
    "TotalImportofCmdbyReporter",
    "AvgUnitPriceofExporterToWorld",
    "TotalExportofCmdbyPartner",
    "Partner_Revealed_Comparative_Advantage",
    "Theil_Exporter_Concentration",
    "Theil_Importer_Concentration",
    "Trade_Complementarity",
    "HS4_Basket_Distance"
)



################################################################################
# Adding Potential Alberta Trades
alberta_data = Trade_data5.filter(F.col("exporter") == 9999)
all_alberta_hsCodes = (
    alberta_data.select("hsCode").distinct()
)
all_countries = (
    Trade_data5
    .select("importer").distinct()
    .filter(F.col("importer") != 124)
)
all_years = Trade_data5.select("year").distinct()
alberta_potential_trades = (
    all_years.crossJoin(all_countries)
             .crossJoin(all_alberta_hsCodes)
)
alberta_potential_trades = alberta_potential_trades.withColumn("exporter", F.lit(9999))
alberta_potential_trades = alberta_potential_trades.orderBy("year", "importer", "hsCode")
#print("Alberta Potential Trades:", alberta_potential_trades.count())
##print("From now on the total rows of this dataframe should not change!")
# Throught the next steps, the size of the dataframe should not chnage.
# --- 1. Join alberta_data on (year, importer, hsCode, exporter) ---
cols = [
    "year", "importer", "hsCode", "exporter",
    "value", "AvgUnitPrice", "Trade_Complementarity", "HS4_Basket_Distance"
]

alberta_potential_trades = (
    alberta_potential_trades
    .join(
        alberta_data.select(cols),
        on=["year", "importer", "hsCode", "exporter"],
        how="left"
    )
)

#print("Alberta Potential Trades:", alberta_potential_trades.count())


# --- 2. Join exporter‑world info ---
cols = [
    "year", "hsCode", "exporter",
    "AvgUnitPriceofExporterToWorld",
    "TotalExportofCmdbyPartner",
    "Partner_Revealed_Comparative_Advantage"
]

alberta_potential_trades = (
    alberta_potential_trades
    .join(
        Trade_data5.select(cols).distinct(),
        on=["year", "hsCode", "exporter"],
        how="left"
    )
)

#print("Alberta Potential Trades:", alberta_potential_trades.count())


# --- 3. Join exporter Theil concentration ---
cols = ["year", "exporter", "Theil_Exporter_Concentration"]

alberta_potential_trades = (
    alberta_potential_trades
    .join(
        Trade_data5.select(cols).distinct(),
        on=["year", "exporter"],
        how="left"
    )
)

#print("Alberta Potential Trades:", alberta_potential_trades.count())


# --- 4. Join importer Theil concentration ---
cols = ["year", "importer", "Theil_Importer_Concentration"]

alberta_potential_trades = (
    alberta_potential_trades
    .join(
        Trade_data5.select(cols).distinct(),
        on=["year", "importer"],
        how="left"
    )
)

#print("Alberta Potential Trades:", alberta_potential_trades.count())


# --- 5. Join importer‑world info ---
cols = [
    "year", "importer", "hsCode",
    "AvgUnitPriceofImporterFromWorld",
    "TotalImportofCmdbyReporter"
]

alberta_potential_trades = (
    alberta_potential_trades
    .join(
        Trade_data5.select(cols).distinct(),
        on=["year", "importer", "hsCode"],
        how="left"
    )
)

#print("Alberta Potential Trades:", alberta_potential_trades.count())


# --- 6. Select final columns in the same order as Trade_data5 ---
alberta_potential_trades = alberta_potential_trades.select(*Trade_data5.columns)

# I drop the rows where TotalImportofCmdbyReporter is missing which means those countries did not import at all. So, there is no potential trade to be made:
# Also drop the rows where TotalExportofCmdbyPartner is missing which means Alberta did not export at all.
#print("Alberta Potential Trades:", alberta_potential_trades.count())
#print("now some rows are dropped:")
alberta_potential_trades = alberta_potential_trades.filter(
    F.col("TotalImportofCmdbyReporter").isNotNull()
)
alberta_potential_trades = alberta_potential_trades.filter(
    F.col("TotalExportofCmdbyPartner").isNotNull()
)
#print("Alberta Potential Trades:", alberta_potential_trades.count())

Trade_data5 = Trade_data5.filter(F.col("exporter") != 9999)
Trade_data6 = Trade_data5.unionByName(alberta_potential_trades)
#print("Trade_data6:", Trade_data6.count())

############################################################################
############################################################################
############################################################################
# Check next lines
# comment all the prints for faster execution
# add HS compoimentarity variable



In [0]:
root_path = f"{model_parameters['root_path']}"
tmp_path = f"{root_path}/tmp_export"

# Step 1: write to temp folder
Trade_data6.coalesce(1).write \
    .option("header", True) \
    .mode("overwrite") \
    .csv(tmp_path)

# Step 2: find the part file
part_file = [f for f in dbutils.fs.ls(tmp_path) if f.name.startswith("part-")][0].path

# Step 3: define final path
final_path = f"{root_path}/1- CEPII_Processed_HS4_{model_parameters['Start_year']}_{model_parameters['End_year']}.csv"

# Step 4: move/rename
dbutils.fs.mv(part_file, final_path)

# Step 5: clean up temp folder
dbutils.fs.rm(tmp_path, recurse=True)


In [0]:
# data.show(10, truncate=False)
# data.describe().show()
# data.filter((F.col("year")== 2023) & (F.col("HS4")== 2805) & (F.col("exporter")== 58)).show(10, truncate=False)